In [2]:
import numpy as np
import matplotlib as mpl
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
import os
import pickle
from tqdm.auto import tqdm
from scipy.ndimage import gaussian_filter
import seaborn as sns
import matplotlib.patches as mpatches
from scipy.stats import wilcoxon

from utils_load_data import *
from utils_trajectories import *


## parameters : 


dt = 0.1
t_pre, t_post = 0.8,0.3 #s
time = np.arange(-t_pre, t_post + dt, dt)
save_directory = '/home/felicie/Documents/Figures/'

session_type = 'playback'
spike_sorted = True

In [ ]:
#plot parameters
mpl.rcdefaults()


plt.rcParams.update({
    'font.size':        7,
    'axes.linewidth':   0.5,
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'xtick.major.width':0.5,
    'ytick.major.width':0.5,
    'xtick.major.size': 2,
    'ytick.major.size': 2,
    'xtick.direction':  'out',
    'ytick.direction':  'out',
    'pdf.fonttype':     42,   # editable text in Illustrator
    'ps.fonttype':      42,
})
C_TRACK = 'red'
C_PB = 'black'
C_MOCK = 'purple'

# for mapping change : same colors, add linestyle ='-.

## Data import

In [3]:
"ss_hs_{headstage_oftqdm_interest}_a1"

#files = [ 'NAPOLEON_hs_0','NAPOLEON_hs_1', 'HERCULE_hs_0', 'MMELOIK_hs_1','FETA_hs_0', 'SKIEUR_hs_0', 'SKIEUR_hs_1']
files = ['SKIEUR_hs_0', 'SKIEUR_hs_1' ]

n_data_all = []
f_data_all = []

for file in files:
    # Load neural data
    if spike_sorted:
        path = f"/auto/data6/eTheremin/{file}_{session_type}_{dt}_data_ss"
    else:
        path = f"/auto/data6/eTheremin/{file}_{session_type}_{dt}_data"

    with open(path, "rb") as fp:
        n_data_s = pickle.load(fp)
        n_data_all.append(n_data_s)
    
    # Load features
    if spike_sorted:
        path_feat = f"/auto/data6/eTheremin/{file}_{session_type}_{dt}_feature_ss"
    else:
        path_feat = f"/auto/data6/eTheremin/{file}_{session_type}_{dt}_feature"

    with open(path_feat, "rb") as fp:
        f_data_s = pickle.load(fp)
        f_data_all.append(f_data_s)

FileNotFoundError: [Errno 2] No such file or directory: '/auto/data6/eTheremin/SKIEUR_hs_0_playback_0.1_data_ss'

In [ ]:
for n_data_s in n_data_all :
    n_data_s = smooth_data(n_data_s, sigma =1)  # smooth neural data 
    n_data_s = remove_average(n_data_s) # remove mean for each unit

n_data_reorganised, f_data_reorganised = re_organise_data(n_data_all, f_data_all) # organise data from first sessions to last sessions (all animals mixed)

Get psth

In [ ]:
n_pre = None # None if we dont remove baseline, otherwise size of the baseline. 
all_traj_track = []
all_traj_track_p, all_traj_track_m = [],[]

all_traj_pb = []
all_traj_pb_p, all_traj_pb_m = [],[]

all_traj_mock = []
all_traj_mock_p, all_traj_mock_m = [],[]


all_traj_track, all_traj_pb, all_traj_mock, all_traj_track_p,all_traj_track_m, all_traj_pb_p,all_traj_pb_m, all_traj_mock_p,all_traj_mock_m = extract_traj(n_data_reorganised,f_data_reorganised, t_pre, t_post, dt, overlap_thresh=np.inf, n_pre = n_pre, full=True)
# Number of bins allowed to overlap between two trajectories

all_traj_tracking = np.concatenate(all_traj_track, axis=1)  # [window_size, total_neurons]
mean_traj_track   = np.nanmean(all_traj_tracking, axis=1)
sem_traj_track    = np.nanstd(all_traj_tracking, axis=1) / np.sqrt(np.sum(~np.isnan(all_traj_tracking), axis=1))

all_traj_playback = np.concatenate(all_traj_pb, axis=1)
mean_traj_pb      = np.nanmean(all_traj_playback, axis=1)
sem_traj_pb       = np.nanstd(all_traj_playback, axis=1) / np.sqrt(np.sum(~np.isnan(all_traj_playback), axis=1))

Find trajectories per frequency

In [ ]:
n_pre = int(t_pre/dt-1)
all_traj_track = []
all_traj_track_p, all_traj_track_m = [],[]

all_traj_pb = []
all_traj_pb_p, all_traj_pb_m = [],[]

all_traj_mock = []
all_traj_mock_p, all_traj_mock_m = [],[]


all_traj_track, all_traj_pb, all_traj_mock, all_traj_track_p,all_traj_track_m, all_traj_pb_p,all_traj_pb_m, all_traj_mock_p,all_traj_mock_m = (n_data_reorganised,f_data_reorganised, t_pre, t_post, dt, f_min = 1000, f_max = 4000, overlap_thresh = np.inf, full=True)

# all_traj_track : sessions x frequencies x time x neurons. 